### Выбор модели астрометрической редукции.

In [1]:
from math import *
import numpy as np
from scipy import stats
from scipy.optimize import curve_fit

import matplotlib.pyplot as plt
%matplotlib widget

Сформируем ансамбль опорных звезд, случайно распределенных на снимке, задав их пиксельные и тангенциальные координаты

In [2]:
N = 100 # число звезд

err = 0.01# стандартная ошибка измерения пиксельных координат
# измеренные координаты в пикселях
x = 2000*np.random.random(N)
y = 2000*np.random.random(N)
scale = 5.0# масштаб arcsec/pix
alpha = radians(33.5) # угол поворота системы пиксельных координат относительно круга склонений deg
x0,y0 = 1000,1000# x,y-координаты нульпункта начала отсчета пиксельных координат относительно центра проекции (оптического центра)

# тангенциальные координаты в arcsec
ksi = ((x-x0)*cos(alpha)+(y-y0)*sin(alpha))*scale
eta = (-(x-x0)*sin(alpha)+(y-y0)*cos(alpha))*scale

# внесем случайные ошибки в пиксельные координаты
x+=np.random.normal(0,err,N)
y+=np.random.normal(0,err,N)

# рабочее поле arcmin
fov = scale*2000/60
print('fov = %5.1f arcmin'%fov)


fov = 166.7 arcmin


Выполним астрометрическую редукцию - применим модель шести постоянных (МНК)

In [3]:
#формируем матрицу плана. Каждая строчка [1 x y] 
C = np.ones((N,3))
C[:,1]=x
C[:,2]=y
#решаем уравнения
Zx = np.dot(np.linalg.inv(np.dot(C.T, C)), np.dot(C.T, ksi))
Zy = np.dot(np.linalg.inv(np.dot(C.T, C)), np.dot(C.T, eta))
#считаем невязки
r_ksi = ksi - np.dot(C,Zx)
r_eta = eta - np.dot(C,Zy)
#ошибки единицы веса характеризуют точность измерений
uwe_ksi = sqrt(np.dot(r_ksi.T,r_ksi)/(N-3))
uwe_eta = sqrt(np.dot(r_eta.T,r_eta)/(N-3))
print('uwe_ksi = %7.3f arcsec,uwe_eta = %7.3f arcsec'%(uwe_ksi,uwe_eta))

uwe_ksi =   0.047 arcsec,uwe_eta =   0.051 arcsec


In [4]:
# функции преобразования координат
def tangential_coordinates(ra,dec,RA,DEC):
    ksi = cos(dec)*sin(ra-RA)/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    eta = (sin(dec)*cos(DEC)-cos(dec)*sin(DEC)*cos(ra-RA))/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    return ksi,eta

def equatorial_coordinates(ksi,eta,RA,Dec):
    x,y,z = np.dot(np.array([[-sin(RA),-cos(RA) * sin(Dec),cos(RA) * cos(Dec)],
                             [cos(RA),-sin(RA) * sin(Dec),sin(RA) * cos(Dec)],
                             [0,cos(Dec),sin(Dec)]]),
                   np.array([ksi,eta,1]))/sqrt(1+ksi*ksi+eta*eta)
    ra = atan2(y,x)
    dec = atan2(z,sqrt(x*x+y*y))
    if(ra<0):
        ra+=2*pi
    return ra,dec

In [5]:
# будем считать, что координаты оптического центра 0,0 (на самом деле в данном примере их можно задать произвольно)
# теперь вычислим экваториальные координаты всех звезд 
ra,dec = np.zeros(N),np.zeros(N)
for k in range(N):
    ra[k],dec[k] = equatorial_coordinates(radians(ksi[k]/3600),radians(eta[k]/3600),radians(0),radians(0))
# теперь изменим оптический центр путем сдвига по склонению на 40 arcmin (четверть поля) и посчитаем новые тангенциальные координаты
ksi_e,eta_e = np.zeros(N),np.zeros(N)
for k in range(N):
    ksi_e[k],eta_e[k] = tangential_coordinates(ra[k],dec[k],radians(20.0/60.0),radians(40.0/60.0))
ksi_e = 3600*np.degrees(ksi_e)
eta_e = 3600*np.degrees(eta_e)

In [6]:
# снова выполним астрометрическую редукцию
Zx = np.dot(np.linalg.inv(np.dot(C.T, C)), np.dot(C.T, ksi_e))
Zy = np.dot(np.linalg.inv(np.dot(C.T, C)), np.dot(C.T, eta_e))
#считаем невязки
r_ksi = ksi_e - np.dot(C,Zx)
r_eta = eta_e - np.dot(C,Zy)
#ошибки единицы 
uwe_ksi = sqrt(np.dot(r_ksi.T,r_ksi)/(N-3))
uwe_eta = sqrt(np.dot(r_eta.T,r_eta)/(N-3))
print('uwe_ksi = %7.3f arcsec,uwe_eta = %7.3f arcsec'%(uwe_ksi,uwe_eta))

uwe_ksi =   0.533 arcsec,uwe_eta =   0.457 arcsec


Мы видим, что ошибка в определении центра проекции серьезно влияет на результат редукции. Допустим, что мы ничего не знаем о том, что у нас присутствует ошибка в определении оптического центра (или имеется наклон плоскости детектора к оптической оси). Первое, что стоит сделать, это посмотреть на то, что происходит с невязками. Есть ли зависимость невязок от координат. Например, можно построить векторное поле невязок (Field Distortion Pattern = FDP)

In [7]:
# строим поле невязок, разбивая всю область снимка на квадраты разумного размера (чтобы помещалось достаточно звезд для осреднения)
Np = 5
lim1,lim2 = -5000,5000
bin2d = np.linspace(lim1,lim2,Np+1)
retX = stats.binned_statistic_2d(ksi, eta, r_ksi, statistic='mean', bins=[bin2d, bin2d])
retY = stats.binned_statistic_2d(ksi, eta, r_eta, statistic='mean', bins=[bin2d, bin2d])

# шаг сетки
step = (lim2-lim1)/Np
# центры квадратов
X,Y = np.mgrid[lim1+step/2:lim2-step/2:complex(0,Np),
                lim1+step/2:lim2-step/2:complex(0,Np)]
# превращяем все массивы в одномерные
xx,yy = X.reshape(Np**2),Y.reshape(Np**2)
dx,dy = retX[0].reshape(Np**2),retY[0].reshape(Np**2)

In [8]:
# рисуем векторное поле
fig, ax = plt.subplots(figsize=(2,2), dpi=300)
ax.set_aspect('equal')
q = plt.quiver(xx,yy,dx,dy,scale=10,color='blue', width=0.005)
ax.quiverkey(q, X=0.98, Y=1.03, U=0.5,label='0.5 arcsec', labelpos='N', angle = 00, labelsep = 0.05,fontproperties={'size': 4})

# plt.scatter(X,Y, c='gray', s=2, zorder=1)
plt.xlim(-5000,5000)
plt.ylim(-5000,5000)
ax.xaxis.set_tick_params(labelsize=5)
ax.yaxis.set_tick_params(labelsize=5)
plt.xlabel('$\\xi$ (arcsec)', fontsize=4)
plt.ylabel('$\\eta$ (arcsec)', fontsize=4)
plt.tight_layout()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

При достаточном числе опорных звезд мы видим, что имеется явная хависимость невязок от координат. Это указывает на то, что модель редукции подобрана некорректно. Теперь надо решить, как ее изменить. Можно пробовать разные варианты. Например, предположить, что есть наклон. Мы знаем, как тогда будет выглядеть модель редукции. Если ошибки единицы веса получатся сопоставимыми с ошибками измерения пиксельных координат, то это будет означать, что модель подобрана удачно.

In [9]:
#формируем матрицу плана. Каждая строчка [1 x y x**2 xy] 
Cx = np.ones((N,5))
Cx[:,1],Cx[:,2],Cx[:,3],Cx[:,4]=x,y,x**2,x*y
#формируем матрицу плана. Каждая строчка [1 x y xy y**2] 
Cy = np.ones((N,5))
Cy[:,1],Cy[:,2],Cy[:,3],Cy[:,4]=x,y,x*y,y**2
#решаем уравнения
Zx = np.dot(np.linalg.inv(np.dot(Cx.T, Cx)), np.dot(Cx.T, ksi_e))
Zy = np.dot(np.linalg.inv(np.dot(Cy.T, Cy)), np.dot(Cy.T, eta_e))
#считаем невязки
r_ksi = ksi_e - np.dot(Cx,Zx)
r_eta = eta_e - np.dot(Cy,Zy)
#ошибки единицы веса характеризуют точность измерений
uwe_ksi = sqrt(np.dot(r_ksi.T,r_ksi)/(N-3))
uwe_eta = sqrt(np.dot(r_eta.T,r_eta)/(N-3))
print('uwe_ksi = %7.3f arcsec,uwe_eta = %7.3f arcsec'%(uwe_ksi,uwe_eta))

uwe_ksi =   0.239 arcsec,uwe_eta =   0.053 arcsec


Видим, что модель восьми постоянных сработала успешно. И это линейная модель, которая работает только если угол поворота между осями пиксельных и тангенциальных координат будет малым (несколько градусов). На практике ориентировка может быть какой угодно. Поэтому придется использовать полную модель, учитывающую наклонность (ошибку определения оптического центра).

In [10]:
# определяем модель в виде функции
def fmodel(xdata,*p):
    cx,ax,bx,cy,ay,by,d,e,f = p
    Nd = int(np.size(xdata)/2)
    x,y = xdata[0:Nd],xdata[Nd:2*Nd] 
    return np.hstack(((ax*x+bx*y+cx)/(d*x+e*y+f), (ay*x+by*y+cy)/(d*x+e*y+f)))
# ищем решение методом Левенберга-Марквардта 
p,covp = curve_fit(f=fmodel, p0=np.array([Zx[0],Zx[1],Zx[2],Zy[0],Zy[1],Zy[2],0.0,0.0,1]),\
                   xdata = np.hstack((x,y)),\
                   ydata = np.hstack((ksi_e,eta_e)))
r = np.hstack((ksi_e,eta_e)) - fmodel(np.hstack((x,y)),*p)
r_ksi,r_eta = r[0:N],r[N:2*N]
uwe = sqrt(np.dot(r.T,r)/(N-9))
print('uwe = %7.3f arcsec'%uwe)
uwe_ksi = sqrt(np.dot(r_ksi.T,r_ksi)/(N-6))
uwe_eta = sqrt(np.dot(r_eta.T,r_eta)/(N-6))
print('uwe_ksi = %7.3f arcsec,uwe_eta = %7.3f arcsec'%(uwe_ksi,uwe_eta))

uwe =   0.071 arcsec
uwe_ksi =   0.048 arcsec,uwe_eta =   0.051 arcsec


Видно, что данная модель является полной (учитывает все систематические эффекты) для разных углов поворота. Но на практике нередко идут другим путем. Заниматься с каждым кадром на таком уровне можно только тогда, когда их мало (например, съемка интересующей Вас области с борта космического телескопа). На практике, когда звезд достаточно много рационально действовать так, как показано ниже.
$$\xi = \sum_{i,j}^{i+j \leq n} a_{i,j}x^i y^j$$

In [11]:
# задаем порядок модели
n=3
# вычисляем число постоянных
Q = int((n+1)*(n+2)/2)
C = np.ones((N, Q))
q = 0
for i in range(n + 1):
    for j in range(n + 1):
        if (i + j <= n):
            C[:, q] = (x ** i) * (y ** j)
            q += 1
# снова выполним астрометрическую редукцию
Zx = np.dot(np.linalg.inv(np.dot(C.T, C)), np.dot(C.T, ksi_e))
Zy = np.dot(np.linalg.inv(np.dot(C.T, C)), np.dot(C.T, eta_e))
#считаем невязки
r_ksi = ksi_e - np.dot(C,Zx)
r_eta = eta_e - np.dot(C,Zy)
#ошибки единицы 
uwe_ksi = sqrt(np.dot(r_ksi.T,r_ksi)/(N-Q))
uwe_eta = sqrt(np.dot(r_eta.T,r_eta)/(N-Q))
print('uwe_ksi = %7.3f arcsec,uwe_eta = %7.3f arcsec'%(uwe_ksi,uwe_eta))

uwe_ksi =   0.047 arcsec,uwe_eta =   0.048 arcsec


Применение таких моделей сталкивается с одной трудностью. Для высоких порядков может оказаться недостаточно опорных звезд. Поэтому стоит быть внимательными при использовании такого варианта. 

#### Задание 
Принято считать, что модели связи пиксельных и тангенциальных координат описываются гладкими функциями. Поэтому, если вас интересуют координаты одного объекта в кадре и рядом с ним достаточно опорных звезд, то рационально выполнить редукцию методом шести постоянных, но в пределах небольшой области кадра. Попробуйте проверить это, отобрав звезды в области, смещенную на половину размера поля относительно центра по каждой из координат и размером 0.2 от размера поля по обеим координатам.